In [ ]:
# to be used when connecting to Dynap-se via remote server (Zemo)
#model, _ = ut.open_dynapse1(gui=False, select_device=True)

In [ ]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1
import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
import params_all_cores
from params_all_cores import set_params
import time
import importlib
from Activity_plotter import run_gui

In [ ]:
print(samna.__version__)

In [ ]:
# to be used when connecting to Dynap-se locally 
devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

In [ ]:
api = model.get_dynapse1_api()
config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 
param_group_c1 = config1.chips[0].cores[1].parameter_group 
param_group_c2 = config1.chips[0].cores[2].parameter_group 
param_group_c3 = config1.chips[0].cores[3].parameter_group 

In [ ]:
param_list = ["IF_AHTAU_N", "IF_AHTHR_N", "IF_AHW_P", "IF_BUF_P", "IF_DC_P", "IF_NMDA_N", "IF_RFR_N", "IF_TAU1_N", "IF_TAU2_N", "IF_THR_N", "NPDPIE_TAU_F_P", "NPDPIE_TAU_S_P", "NPDPIE_THR_F_P", "NPDPIE_THR_S_P", 
              "NPDPII_TAU_F_P", "NPDPII_TAU_S_P", "NPDPII_THR_F_P", "NPDPII_THR_S_P", "PS_WEIGHT_EXC_F_N", "PS_WEIGHT_EXC_S_N", "PS_WEIGHT_INH_F_N", "PS_WEIGHT_INH_S_N", "PULSE_PWLK_P", "R2R_P"]

In [ ]:
"""for i in param_list:
    print(param_group_c2.param_map[i])"""

LIF encoding

In [ ]:
# ----------------  stimulus: a Gaussian bump ----------------
n_pts     = 1000                 # number of samples
t_end     = 1.0                  # seconds  (→ dt = 1 ms)
t         = np.linspace(0, t_end, n_pts, endpoint=False)
x         = np.linspace(-4, 4, n_pts)
sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
#I_peak    = 30000000e-12              # 1000 pA
I_peak    = 30000000e-12              # 1000 pA

I         = gauss/gauss.max() * I_peak   # injected current (A)

# ----------------  LIF neuron parameters ----------------------
tau_m     = 20e-3                # 20 ms membrane time constant
R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
C_m       = tau_m / R_m
v_rest    = -65e-3               # -65 mV
v_reset   = -65e-3
v_thresh  = -50e-3               # spike threshold
t_ref     = 2e-3                 # 2 ms refractory period
dt        = t_end / n_pts        # simulation time-step (s)

# ----------------  simulation loop ----------------------------
v        = v_rest
next_ok  = 0.0                   # time when refractory ends
v_trace  = np.empty(n_pts)
spikes   = []

for k in range(n_pts):
    if t[k] >= next_ok:          # not in refractory
        dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
        v += dv
        if v >= v_thresh:        # spike!
            spikes.append(t[k])
            v = v_reset
            next_ok = t[k] + t_ref
    v_trace[k] = v

spikes = np.array(spikes)

# ----------------  plots --------------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8,4), sharex=True)

ax1.plot(t*1e3, I*1e12)
ax1.set_ylabel('I (pA)'); ax1.set_title('Input current'); ax1.grid()

ax2.plot(t*1e3, v_trace*1e3)
ax2.scatter(spikes*1e3, np.full(len(spikes), (v_thresh+5e-3)*1e3),
            marker='|', color='red', label='spikes')
ax2.set_xlabel('time (ms)')
ax2.set_ylabel('V (mV)')
ax2.set_title('LIF membrane potential & spike train')
ax2.legend(); ax2.grid()
plt.tight_layout(); plt.show()

In [ ]:
spike_times_all = spikes#*1e3

In [ ]:
len(spike_times_all)

In [ ]:
spike_times_all

In [ ]:
spike_ids = np.full(len(spikes), 1)

In [ ]:
spikegen_ids = [(0, 1, n) for n in range(10)]

In [ ]:
print(spikegen_ids)

In [ ]:
# double checking connections
"""
# create neuron populations in the ring
chip = 0
core = 1
npop = 1
NBINS = 10

# Create neuron populations for each band in core 0
ring_pops = [
    [Neuron(chip, core, j) for j in range((i * npop), ((i + 1) * npop))]
    for i in range(NBINS)
]

OFFSET_1 = (-1, 1)          # ±3 bins wide “hat”

print("excitatory connections to first neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                print(pre, post)
                print(pre, post)
                print(pre, post)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors
print(" ")
print("excitatory connections to second neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                print(pre, post)
                print(pre, post)

OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
    # todo excitatory connections to third neighbors
print(" ")
print("excitatory connections to third neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                print(pre, post)

    # todo inhibitory connections to all of the other pops
OFFSET_inh = (-5, -4, 4, 5)          # all of the pops that are not being excited
print(" ")
print("inhibitory connections to all of the other pops")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                print(pre, post)
"""

In [ ]:
import params_all_cores

importlib.reload(params_all_cores)
config1 = model.get_configuration()

In [ ]:
timesleep = 30

In [ ]:
pop_nr = 9

In [ ]:
###############################################################################
# 1)  Declare the set of bad neurons once, in (chip, core, neuron_id) format
###############################################################################
BROKEN_NEURONS = {(0, 1, 51), (0, 1, 71), (0, 1, 80), (0, 1, 88), (0, 1, 91)}          #  ⬅️  add more here if needed

def is_ok(chip: int, core: int, nid: int) -> bool:
    """True if this physical neuron should be used."""
    return (chip, core, nid) not in BROKEN_NEURONS


In [ ]:
#importlib.reload(params_all_cores)
#model.update_parameter_group(param_group, 0, 0) # update params based on the params you changed 
import importlib, dynapse1utils as ut

importlib.reload(params_all_cores)

importlib.reload(ut)

config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 

p_E_E   = 1
p_mexican = 1
p_I_I   = 0 #1
p_E_I   = 0 #0.1
p_I_E   = 0 #0.4

# initiate network 
net_gen = NetworkGenerator()
net_gen.clear_network()

# create spikegens, one per ring attractor neural pop 
spikegen_ids = [(0, 0, n) for n in range(10)]
#isi_spikegen = Neuron(0, 0, 200, True)
#isi_neuron = Neuron(0, 1, 200)

spikegens = []
for spikegen_id in spikegen_ids:
    spikegens.append(Neuron(spikegen_id[0], spikegen_id[1],spikegen_id[2], True))

#spikegens.append(isi_spikegen)

# Create ring neuron populations
chip = 0
core = 1
npop = 4
NBINS = 10
offset_nr = 48

"""ring_pops = [
    [Neuron(chip, core, j) for j in range(offset_nr + (i * npop), offset_nr + ((i + 1) * npop))]
    for i in range(NBINS)
]"""

"""ring_pops = [
    [Neuron(chip, core, j)
     for j in range(offset_nr + i*npop, offset_nr + (i+1)*npop)
     if is_ok(chip, core, j)]                       # ⬅️ skip bad ones
    for i in range(NBINS)
]"""

ring_pops = []
next_id = offset_nr
for _ in range(NBINS):
    pop = []
    while len(pop) < npop:
        if is_ok(chip, core, next_id):
            pop.append(Neuron(chip, core, next_id))
        next_id += 1
    ring_pops.append(pop)


for nr, pop in enumerate(ring_pops):
    print("Ring pop nr.: ", nr)
    print(pop)

print("ring populations: ", ring_pops)    

# create inhibitory population that connects to all other pops
core_inh = 2
start_inh_neuron = 4
npop_inh = 4
pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]
pop_inhibitory
    
"""# create one population in another core that is stimulated by DC so that it's always active
core_dc = 3
start_dc_neuron = 249
npop_dc = 4
pop_dc = [Neuron(chip, core_dc, j) for j in range(start_dc_neuron, start_dc_neuron + npop_dc, 1)]"""

"""# Generate array with events every 0.01 s
time_interval = 0.001
isi_array = np.arange(0, timesleep+1, step=time_interval)
isi_spikegen_id = 200"""


# connect each spikegen to a ring attractor pop 
"""for bin_idx, sg in enumerate(spikegens):       
    for exc in ring_pops[bin_idx]:
        net_gen.add_connection(sg, exc, dyn1.Dynapse1SynType.AMPA)"""

print(spikegens[1])
        
"""for i in ring_pops[pop_nr]:
    print(i)
    net_gen.add_connection(spikegens[1], i, dyn1.Dynapse1SynType.AMPA)
    """
    
# SPIKEGEN CONNECTIONS
net_gen.add_connection(spikegens[1], ring_pops[pop_nr][0], dyn1.Dynapse1SynType.AMPA)
net_gen.add_connection(spikegens[1], ring_pops[pop_nr][1], dyn1.Dynapse1SynType.AMPA)
net_gen.add_connection(spikegens[1], ring_pops[pop_nr][2], dyn1.Dynapse1SynType.AMPA)
net_gen.add_connection(spikegens[1], ring_pops[pop_nr][3], dyn1.Dynapse1SynType.AMPA)
    
# connect isi spikegen to isi neuron 
#net_gen.add_connection(isi_spikegen, isi_neuron, dyn1.Dynapse1SynType.AMPA)

# self excitation in each neural population in the ring: (todo determine if this is needed) 
for pop in ring_pops:
    for pre in pop:
        for post in pop:
            if pre is not post and np.random.rand() < p_E_E:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)
                                
# MEXICAN HAT CONNECTIONS
OFFSET_1 = (-1, 1)         
print("excitatory connections to first neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors
print(" ")
print("excitatory connections to second neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)


    #  excitatory connections to third neighbors
OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
print(" ")
print("excitatory connections to third neighbors")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                print(" ")
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

    # todo inhibitory connections to all of the other pops
OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
print(" ")
print("inhibitory connections to all of the other pops")
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

# INH → EXC  (global inhibition pop to all pops in the ring)
for inh in pop_inhibitory:
    for pop in ring_pops:
        for exc in pop:
            net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

# EXC → INH  (drive the global inhibition pop from all pops in the ring)
for pop in ring_pops:
    for exc in pop:
        for inh in pop_inhibitory:
            net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)

# Print the network for verification (optional)
print(net_gen.network)

# make a dynapse1config using the network
new_config = net_gen.make_dynapse1_configuration()

# apply the configuration
model.apply_configuration(new_config)

# Set hardware parameters
set_params(model)

fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 

# get events of selected neurons
"""monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)   
    for pop in ring_pops
    for n   in pop
]"""

monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)
    for pop in ring_pops
    for n   in pop
]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_inhibitory  
])

"""monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_dc  
])"""

#monitored_neurons.extend([(isi_neuron.chip_id, isi_neuron.core_id, isi_neuron.neuron_id)])

print("monitored neurons", monitored_neurons)

"""random_pop = [Neuron(chip, core_inh, j) for j in range(70, 200, 1)]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in random_pop  
])"""

graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
graph.start()

# clear the buffer
sink_node.get_events()

# select the neurons to monitor
print("monitored neurons:", monitored_neurons)
filter_node.set_neurons(monitored_neurons)

api.reset_timestamp()

ut.set_neuron_tau1(model, 0, 0, (7, 255))
ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))
ut.set_neuron_tau1(model, 0, 3, (7, 255))

time.sleep(1)

ut.set_neuron_tau1(model, 0, 0, (4, 50))
ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 50))
ut.set_neuron_tau1(model, 0, 3, (4, 50))


#spike_times_s = spike_times_all  # spike_times_all is in seconds!!!
#spike_times_all = np.concatenate([spike_times_s, isi_array])
spike_times_all #= spike_times_s
print("spike_times_all length", len(spike_times_all))

#spike_ids     = np.asarray(spike_ids,   dtype=np.int32).tolist()
#spike_times_s = np.asarray(spike_times_s, dtype=np.float64).tolist()

#spike_ids_all = np.concatenate([spike_ids, [isi_spikegen_id] * len(isi_array)])
spike_ids_all = spike_ids
print("spike_ids_all length", len(spike_ids_all))


# Sort input events in time order
sort_indices = np.argsort(spike_times_all)
all_spike_times = spike_times_all[sort_indices]
all_spike_ids = spike_ids_all[sort_indices]

print("all_spike_times length", len(all_spike_times))
print("all_spike_ids length", len(all_spike_ids))

#time.sleep(2)

ut.set_fpga_spike_gen(
    fpga_spike_gen,
    all_spike_times,
    all_spike_ids,
    target_chips=[0] * len(all_spike_ids),
    isi_base=900,
    repeat_mode=False)

fpga_spike_gen.start()

#time.sleep(D_post_stim)
time.sleep(timesleep)

fpga_spike_gen.stop()

graph.stop()
events = sink_node.get_events()

#run_gui(sink_node, refresh_rate=500, y_range = [0, 255])

# process the events
print(len(events), "events.")

NN = 256 # neurons in 1 core
spike_id = []
spike_t  = []
for evt in events:
    #spike_id.append(evt.core_id*NN + evt.neuron_id)
    spike_id.append(evt.neuron_id)
    spike_t.append(evt.timestamp*1e-6)
spike_id = np.array(spike_id)
spike_t  = np.array(spike_t) 

# ─────────────────────────  POPULATION–RATE ANALYSIS  ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# 1.  masks for the two pools ---------------------------------------------------
ring_ids = {n.neuron_id for pop in ring_pops     for n in pop}
inh_ids  = {n.neuron_id for n in pop_inhibitory}

ring_mask = np.isin(spike_id, list(ring_ids))
inh_mask  = np.isin(spike_id, list(inh_ids))

# 2.  binning -------------------------------------------------------------------
bin_size  = 0.050                         # 50 ms  (→ 20 Hz sampling)
t_edges   = np.arange(0, timesleep + bin_size, bin_size)
t_centers = t_edges[:-1] + bin_size/2

# 3.  histogram → population rate (Hz)  -----------------------------------------
def pop_rate(mask):
    counts, _ = np.histogram(spike_t[mask], bins=t_edges)
    return counts / bin_size                         # spikes s⁻¹ (Hz)

ring_rate_hz = pop_rate(ring_mask)
inh_rate_hz  = pop_rate(inh_mask)

print(f"Mean excitatory (ring) rate : {ring_rate_hz.mean():.2f} Hz")
print(f"Mean inhibitory rate       : {inh_rate_hz.mean():.2f} Hz")

# 4.  plotting ------------------------------------------------------------------
fig, (ax_raster, ax_ring, ax_inh) = plt.subplots(
        3, 1, sharex=True, figsize=(10, 14),
        gridspec_kw={'height_ratios': [4, 1, 1]})

# link only the two population-rate axes
ax_inh.sharey(ax_ring)
#ax_inh.tick_params(labelleft=False)        # hide duplicate y-ticks on ax_inh

# ─ Raster ─────────────────────────────────────────────────────────────────────
ax_raster.scatter(spike_t, spike_id, s=4)
ax_raster.set_ylabel("Neuron ID")
ax_raster.set_xlim(-2, timesleep)
#ax_raster.set_ylim(0, 50)
ax_raster.set_yticks(np.arange(0, spike_id.max() + 1, 3))
ax_raster.set_title("Raster and population firing rates")

# ─ Excitatory population rate ────────────────────────────────────────────────
ax_ring.plot(t_centers, ring_rate_hz, lw=1.5)
ax_ring.set_ylabel('Excit. rate (Hz)')
ax_ring.grid(alpha=.3)

# ─ Inhibitory population rate ────────────────────────────────────────────────
ax_inh.plot(t_centers, inh_rate_hz, lw=1.5, color='tab:red')
ax_inh.set_xlabel('Time (s)')
ax_inh.set_ylabel('Inhib. rate (Hz)')
ax_inh.grid(alpha=.3)

# Aggregate ring neurons by bin
NBINS = len(ring_pops)
bin_firing = np.zeros(NBINS)

for i, pop in enumerate(ring_pops):
    neuron_ids = [n.neuron_id for n in pop]
    mask = np.isin(spike_id, neuron_ids)
    bin_firing[i] = np.sum(mask) / timesleep  # average firing rate (Hz) in that bin

# Map each bin to angle
angles = np.linspace(0, 2 * np.pi, NBINS, endpoint=False)

# Plot
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, polar=True)

# Normalize color
cmap = plt.cm.hot
norm = plt.Normalize(vmin=0, vmax=np.max(bin_firing))
colors = cmap(norm(bin_firing))

bars = ax.bar(angles, bin_firing, width=2*np.pi/NBINS, bottom=0.0, color=colors, edgecolor='k')

ax.set_title("Ring activity snapshot (mean firing rate per bin)")
plt.show()


In [ ]:
ring_pops[pop_nr][0]

In [ ]:
# ─────────────────────────  POPULATION–RATE ANALYSIS  ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# 1.  masks for the two pools ---------------------------------------------------
ring_ids = {n.neuron_id for pop in ring_pops     for n in pop}
inh_ids  = {n.neuron_id for n in pop_inhibitory}

ring_mask = np.isin(spike_id, list(ring_ids))
inh_mask  = np.isin(spike_id, list(inh_ids))

# 2.  binning -------------------------------------------------------------------
bin_size  = 0.050                         # 50 ms  (→ 20 Hz sampling)
t_edges   = np.arange(0, timesleep + bin_size, bin_size)
t_centers = t_edges[:-1] + bin_size/2

# 3.  histogram → population rate (Hz)  -----------------------------------------
def pop_rate(mask):
    counts, _ = np.histogram(spike_t[mask], bins=t_edges)
    return counts / bin_size                         # spikes s⁻¹ (Hz)

ring_rate_hz = pop_rate(ring_mask)
inh_rate_hz  = pop_rate(inh_mask)

print(f"Mean excitatory (ring) rate : {ring_rate_hz.mean():.2f} Hz")
print(f"Mean inhibitory rate       : {inh_rate_hz.mean():.2f} Hz")

# 4.  plotting ------------------------------------------------------------------
fig, (ax_raster, ax_ring, ax_inh) = plt.subplots(
        3, 1, sharex=True, figsize=(10, 12),
        gridspec_kw={'height_ratios': [1.5, 1, 1]})

# link only the two population-rate axes
ax_inh.sharey(ax_ring)
#ax_inh.tick_params(labelleft=False)        # hide duplicate y-ticks on ax_inh

# ─ Raster ─────────────────────────────────────────────────────────────────────
ax_raster.scatter(spike_t, spike_id, s=4)
ax_raster.set_ylabel("Neuron ID")
ax_raster.set_xlim(0, timesleep)
ax_raster.set_ylim(0, 50)
#ax_raster.set_yticks(np.arange(0, spike_id.max() + 1, 2))
ax_raster.set_title("Raster and population firing rates")

# ─ Excitatory population rate ────────────────────────────────────────────────
ax_ring.plot(t_centers, ring_rate_hz, lw=1.5)
ax_ring.set_ylabel('Excit. rate (Hz)')
ax_ring.grid(alpha=.3)

# ─ Inhibitory population rate ────────────────────────────────────────────────
ax_inh.plot(t_centers, inh_rate_hz, lw=1.5, color='tab:red')
ax_inh.set_xlabel('Time (s)')
ax_inh.set_ylabel('Inhib. rate (Hz)')
ax_inh.grid(alpha=.3)

In [ ]:
chip_id = 0
neuron_id = 256 + 42

api.monitor_neuron(chip_id, neuron_id)

In [ ]:
#ut.close_dynapse1(model)